# 09 — Copy Ablation Runs to Final Rerun Directories

**Purpose**: one-time utility to copy completed ablation runs into the final
rerun output roots (`runs_final_v1/` and `runs_inca_final_v1/`), renaming
them with clean experiment names that match notebooks 07 and 08.

> **Run this BEFORE notebooks 07 and 08.** After running, notebooks 07 and
> 08 will detect these copied runs via their 3-case skip-logic (Case A) and
> skip re-training, saving hours of compute.

## What gets copied

### UNM — 5 experiments × 3 seeds = 15 runs

| Source (in `runs_lambda_ablation/`) | Destination (in `runs_final_v1/`) |
|---|---|
| `supervised_p40` | `supervised` |
| `semi_all_lateral_lam005_s15p40` | `semi_all_lateral` |
| `mean_teacher_all_lateral_lam005_s15p40` | `mean_teacher_all_lateral` |
| `semi_r10_lam005_s15p40` | `semi_r10` |
| `mean_teacher_r10_lam005_s15p40` | `mean_teacher_r10` |

### INCA v2 — 3 experiments × 3 seeds = 9 runs

| Source (in `runs_inca_lambda_ablation/`) | Destination (in `runs_inca_final_v1/`) |
|---|---|
| `supervised_inca_p40` | `supervised_inca` |
| `semi_inca_all_lateral_lam005_s7p40` | `semi_inca_all_lateral` |
| `mean_teacher_inca_all_lateral_lam005_s7p40` | `mean_teacher_inca_all_lateral` |

**Total: 24 runs copied.**

## What this notebook does (per seed folder)

1. Verify the source has `best_model.pt` + `test_metrics.csv` + `*_run_report.json` (skip with warning if any missing)
2. Skip if the destination already has `best_model.pt` (non-destructive)
3. Copy the entire seed folder with `shutil.copytree` (all artifacts: model, CSVs, predictions, etc.)
4. Inside the copied folder, find the `*_run_report.json`, read it, modify only these **3 fields**:
   - `run_identity.experiment_name` → clean name
   - `run_identity.exp_dir` → new destination path
   - `training_summary.exp_dir` → new destination path
5. Save the modified report under the new clean filename (e.g. `supervised_seed_0_run_report.json`) and delete the old-named file

## Safety guarantees

- **Sources are read-only**: `runs_lambda_ablation/` and `runs_inca_lambda_ablation/` are never modified or deleted
- **Non-destructive**: if the destination already has `best_model.pt`, the script skips it
- **No partial overwrites**: `shutil.copytree` fails fast if the destination folder exists, and we pre-check for `best_model.pt`
- **`config.json` is NOT modified** inside the copied folders (only the `run_report.json` is patched, and only the 3 fields listed above)
- **No `src/` imports, no `git clone`** — stdlib only (`os`, `shutil`, `json`, `glob`)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import os
import shutil
import json
import glob

BASE = "/content/drive/MyDrive/UNM_vertebras_seg_v3"

UNM_SRC_ROOT = f"{BASE}/runs_lambda_ablation"
UNM_DST_ROOT = f"{BASE}/runs_final_v1"
INCA_SRC_ROOT = f"{BASE}/runs_inca_lambda_ablation"
INCA_DST_ROOT = f"{BASE}/runs_inca_final_v1"

# (source_exp_name, clean_destination_name)
UNM_MAPPING = [
    ("supervised_p40",                            "supervised"),
    ("semi_all_lateral_lam005_s15p40",            "semi_all_lateral"),
    ("mean_teacher_all_lateral_lam005_s15p40",    "mean_teacher_all_lateral"),
    ("semi_r10_lam005_s15p40",                    "semi_r10"),
    ("mean_teacher_r10_lam005_s15p40",            "mean_teacher_r10"),
]

INCA_MAPPING = [
    ("supervised_inca_p40",                                "supervised_inca"),
    ("semi_inca_all_lateral_lam005_s7p40",                 "semi_inca_all_lateral"),
    ("mean_teacher_inca_all_lateral_lam005_s7p40",         "mean_teacher_inca_all_lateral"),
]

SEEDS = [0, 1, 2]

# Counters
n_copied = 0
n_skipped_exists = 0
n_skipped_missing_src = 0
n_skipped_incomplete_src = 0
n_errors = 0


def copy_and_rename(src_exp, clean_exp, src_root, dst_root):
    """Copy 3 seed folders from src_root/src_exp/seed_N to dst_root/clean_exp/seed_N,
    then rename the run_report.json inside the destination."""
    global n_copied, n_skipped_exists, n_skipped_missing_src
    global n_skipped_incomplete_src, n_errors

    for seed in SEEDS:
        src_seed_dir = os.path.join(src_root, src_exp, f"seed_{seed}")
        dst_seed_dir = os.path.join(dst_root, clean_exp, f"seed_{seed}")
        tag = f"{src_exp}/seed_{seed} -> {clean_exp}/seed_{seed}"

        # 1. Source exists?
        if not os.path.isdir(src_seed_dir):
            print(f"  [SKIP missing src] {tag}")
            n_skipped_missing_src += 1
            continue

        # 2. Source has the required artifacts?
        src_best = os.path.join(src_seed_dir, "best_model.pt")
        src_metrics = os.path.join(src_seed_dir, "test_metrics.csv")
        src_reports = glob.glob(os.path.join(src_seed_dir, "*_run_report.json"))

        missing = []
        if not os.path.isfile(src_best):
            missing.append("best_model.pt")
        if not os.path.isfile(src_metrics):
            missing.append("test_metrics.csv")
        if not src_reports:
            missing.append("*_run_report.json")
        if missing:
            print(f"  [SKIP incomplete src] {tag}  (missing: {missing})")
            n_skipped_incomplete_src += 1
            continue

        # 3. Destination safety
        dst_best = os.path.join(dst_seed_dir, "best_model.pt")
        if os.path.isfile(dst_best):
            print(f"  [SKIP dst already has best_model.pt] {tag}")
            n_skipped_exists += 1
            continue

        if os.path.isdir(dst_seed_dir):
            # Destination dir exists but no best_model.pt — weird partial state,
            # don't overwrite (user can investigate manually)
            print(f"  [SKIP dst dir exists partial, leaving alone] {tag}")
            n_errors += 1
            continue

        # 4. Copy
        try:
            os.makedirs(os.path.dirname(dst_seed_dir), exist_ok=True)
            shutil.copytree(src_seed_dir, dst_seed_dir)
        except Exception as exc:
            print(f"  [ERROR copytree] {tag}: {exc}")
            n_errors += 1
            continue

        print(f"  [COPIED] {tag}")
        n_copied += 1

        # 5. Rename / patch run_report.json inside destination
        try:
            dst_reports = glob.glob(os.path.join(dst_seed_dir, "*_run_report.json"))
            if not dst_reports:
                print(f"    WARN: no run_report found after copy")
                continue

            # Prefer the authoritative one (filename matches the source prefix)
            old_prefix = f"{src_exp}_seed_{seed}_"
            authoritative = [r for r in dst_reports if os.path.basename(r).startswith(old_prefix)]
            old_rpt_path = authoritative[0] if authoritative else dst_reports[0]

            with open(old_rpt_path, encoding="utf-8") as f:
                rpt = json.load(f)

            # Modify only the 3 fields
            if isinstance(rpt.get("run_identity"), dict):
                rpt["run_identity"]["experiment_name"] = clean_exp
                rpt["run_identity"]["exp_dir"] = dst_seed_dir
            if isinstance(rpt.get("training_summary"), dict):
                if "exp_dir" in rpt["training_summary"]:
                    rpt["training_summary"]["exp_dir"] = dst_seed_dir

            new_rpt_name = f"{clean_exp}_seed_{seed}_run_report.json"
            new_rpt_path = os.path.join(dst_seed_dir, new_rpt_name)

            with open(new_rpt_path, "w", encoding="utf-8") as f:
                json.dump(rpt, f, indent=2, ensure_ascii=False)

            if os.path.abspath(old_rpt_path) != os.path.abspath(new_rpt_path):
                os.remove(old_rpt_path)
                print(f"    renamed report: {os.path.basename(old_rpt_path)} -> {new_rpt_name}")
            else:
                print(f"    report already had clean name")
        except Exception as exc:
            print(f"    ERROR patching run_report: {exc}")
            n_errors += 1


print("=" * 70)
print("UNM experiments -> runs_final_v1/")
print("=" * 70)
for src, clean in UNM_MAPPING:
    copy_and_rename(src, clean, UNM_SRC_ROOT, UNM_DST_ROOT)

print()
print("=" * 70)
print("INCA v2 experiments -> runs_inca_final_v1/")
print("=" * 70)
for src, clean in INCA_MAPPING:
    copy_and_rename(src, clean, INCA_SRC_ROOT, INCA_DST_ROOT)

print()
print("=" * 70)
print("COPY SUMMARY")
print("=" * 70)
print(f"  Copied:               {n_copied}")
print(f"  Skipped (dst exists): {n_skipped_exists}")
print(f"  Skipped (src missing):{n_skipped_missing_src}")
print(f"  Skipped (incomplete): {n_skipped_incomplete_src}")
print(f"  Errors:               {n_errors}")
_total_seen = n_copied + n_skipped_exists + n_skipped_missing_src + n_skipped_incomplete_src + n_errors
print(f"  Total processed:      {_total_seen}  (expected 24)")


In [ ]:
import os
import json
import glob

BASE = "/content/drive/MyDrive/UNM_vertebras_seg_v3"

UNM_EXPECTED = [
    "supervised",
    "semi_all_lateral",
    "mean_teacher_all_lateral",
    "semi_r10",
    "mean_teacher_r10",
]
INCA_EXPECTED = [
    "supervised_inca",
    "semi_inca_all_lateral",
    "mean_teacher_inca_all_lateral",
]


def verify(label, dst_root, expected):
    print(f"\n=== {label} ===")
    n_total = 0
    n_ok = 0
    n_issues = 0

    for exp in expected:
        exp_dir = os.path.join(dst_root, exp)
        if not os.path.isdir(exp_dir):
            print(f"  [MISSING exp dir] {exp}")
            for _ in [0, 1, 2]:
                n_total += 1
                n_issues += 1
            continue

        for seed in [0, 1, 2]:
            n_total += 1
            seed_dir = os.path.join(exp_dir, f"seed_{seed}")
            issues = []

            if not os.path.isdir(seed_dir):
                print(f"  [MISSING] {exp}/seed_{seed}")
                n_issues += 1
                continue

            if not os.path.isfile(os.path.join(seed_dir, "best_model.pt")):
                issues.append("no best_model.pt")
            if not os.path.isfile(os.path.join(seed_dir, "test_metrics.csv")):
                issues.append("no test_metrics.csv")

            # Check clean-named run_report
            expected_name = f"{exp}_seed_{seed}_run_report.json"
            expected_path = os.path.join(seed_dir, expected_name)

            if not os.path.isfile(expected_path):
                others = glob.glob(os.path.join(seed_dir, "*_run_report.json"))
                if others:
                    issues.append(f"no clean-named report (found: {[os.path.basename(o) for o in others]})")
                else:
                    issues.append("no run_report.json at all")
            else:
                try:
                    with open(expected_path, encoding="utf-8") as f:
                        rpt = json.load(f)
                    inner = (rpt.get("run_identity") or {}).get("experiment_name")
                    if inner != exp:
                        issues.append(f"report experiment_name={inner!r} != {exp!r}")
                except Exception as exc:
                    issues.append(f"report read error: {exc}")

            if issues:
                print(f"  [ISSUES] {exp}/seed_{seed}: {issues}")
                n_issues += 1
            else:
                print(f"  [OK]     {exp}/seed_{seed}")
                n_ok += 1

    return n_total, n_ok, n_issues


def count_exp_dirs(dst_root, expected):
    if not os.path.isdir(dst_root):
        return 0
    found = 0
    for exp in expected:
        if os.path.isdir(os.path.join(dst_root, exp)):
            found += 1
    return found


unm_dirs = count_exp_dirs(f"{BASE}/runs_final_v1", UNM_EXPECTED)
inca_dirs = count_exp_dirs(f"{BASE}/runs_inca_final_v1", INCA_EXPECTED)
print(f"Experiment folders present:")
print(f"  runs_final_v1/        : {unm_dirs}/{len(UNM_EXPECTED)} expected")
print(f"  runs_inca_final_v1/   : {inca_dirs}/{len(INCA_EXPECTED)} expected")

unm_total, unm_ok, unm_issues = verify("UNM (runs_final_v1/)", f"{BASE}/runs_final_v1", UNM_EXPECTED)
inca_total, inca_ok, inca_issues = verify("INCA (runs_inca_final_v1/)", f"{BASE}/runs_inca_final_v1", INCA_EXPECTED)

total = unm_total + inca_total
ok = unm_ok + inca_ok
issues = unm_issues + inca_issues

print()
print("=" * 70)
print("VERIFICATION SUMMARY")
print("=" * 70)
print(f"  Total runs scanned:   {total}  (expected 24)")
print(f"  Verified complete OK: {ok}")
print(f"  With issues:          {issues}")
